In [1]:
!pip install -q transformers requests sentence-transformers

import os
import getpass

if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass.getpass('HF token (read scope): ')


HF token (read scope): ··········


In [2]:
import os
import requests
import time

HF_TOKEN = os.environ["HF_TOKEN"]

API_URL = "https://api-inference.huggingface.co/models/facebook/bart-large-mnli"

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

def hf_zero_shot_api(text, labels):
    payload = {
        "inputs": text,
        "parameters": {
            "candidate_labels": labels
        }
    }

    try:
        r = requests.post(API_URL, headers=headers, json=payload, timeout=30)

        print("Status:", r.status_code)

        # Handle model loading
        if r.status_code == 503:
            return {"error": "Model is loading. Try again in a few seconds."}

        r.raise_for_status()

        return r.json()

    except requests.exceptions.RequestException as e:
        return {"error": str(e)}


resumes = [
    "Built React dashboards for 3 startups",
    "Implemented Spring Boot microservices in Java for fintech app",
    "Trained CNN for image classification using PyTorch, 87% accuracy",
    "Cleaned 100k row dataset using pandas and matplotlib for reports",
    "Wrote SQL queries against PostgreSQL, optimised queries by 10x",
]

labels = [
    "frontend dev",
    "backend dev",
    "data analyst",
    "ML engineer"
]

start = time.time()

for resume in resumes:
    result = hf_zero_shot_api(resume, labels)

    if "error" in result:
        print("Error:", result["error"])
    else:
        top_label = result["labels"][0]
        top_score = result["scores"][0]

        print(f"{resume[:55]:55} -> {top_label} ({top_score:.2f})")

print(f"\nTotal API time: {time.time() - start:.2f}s")

Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/facebook/bart-large-mnli (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x790edf333170>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)"))
Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/facebook/bart-large-mnli (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x790edea2f4d0>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)"))
Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/facebook/bart-large-mnli (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x790edea2fb60>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)"))
Error: HTTPSConnectionPoo

In [3]:
from transformers import pipeline

# This DOWNLOADS the model on first run — ~1.6GB. Be patient.
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

start = time.time()
for r in resumes:
    res = classifier(r, candidate_labels=labels)
    print(f'  {r[:50]:50} -> {res["labels"][0]} ({res["scores"][0]:.2f})')
print(f'\nLocal time (after download): {time.time()-start:.2f}s')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

  Built React dashboards for 3 startups              -> frontend dev (0.88)
  Implemented Spring Boot microservices in Java for  -> backend dev (0.63)
  Trained CNN for image classification using PyTorch -> ML engineer (0.40)
  Cleaned 100k row dataset using pandas and matplotl -> data analyst (0.56)
  Wrote SQL queries against PostgreSQL, optimised qu -> data analyst (0.42)

Local time (after download): 14.09s


In [4]:
sentiment = pipeline('sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english')

# Test data — 5 mock interview answers
answers = [
    'I really enjoyed working on the team and shipped 3 features.',
    'I was the only one writing code; everyone else was slow.',
    'I learned a lot from my mentor and grew technically.',
    "I had to redo most of my teammate's work because it was wrong.",
    'My internship was great — would recommend it to anyone.',
]

print('Sentiment scores:')
for a in answers:
    result = sentiment(a)[0]
    label = result['label']
    score = result['score']
    print(f'  [{label} {score:.2f}] {a[:60]}')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentiment scores:
  [POSITIVE 1.00] I really enjoyed working on the team and shipped 3 features.
  [NEGATIVE 1.00] I was the only one writing code; everyone else was slow.
  [POSITIVE 1.00] I learned a lot from my mentor and grew technically.
  [NEGATIVE 1.00] I had to redo most of my teammate's work because it was wron
  [POSITIVE 1.00] My internship was great — would recommend it to anyone.


In [5]:
import time

def time_call(fn, n_runs=3):
    times = []
    for _ in range(n_runs):
        start = time.time()
        fn()
        times.append(time.time() - start)
    return min(times), sum(times)/len(times)

# Time API call (warm)
def call_api():
    hf_zero_shot_api('Built React dashboards', ['frontend dev', 'backend dev'])
api_min, api_avg = time_call(call_api)

# Time local call (warm)
def call_local():
    classifier('Built React dashboards', candidate_labels=['frontend dev', 'backend dev'])
local_min, local_avg = time_call(call_local)

print(f'Inference timing comparison (3 runs each, after warm-up):')
print(f'  API:   min {api_min:.2f}s | avg {api_avg:.2f}s')
print(f'  Local: min {local_min:.2f}s | avg {local_avg:.2f}s')

Inference timing comparison (3 runs each, after warm-up):
  API:   min 0.01s | avg 0.04s
  Local: min 0.88s | avg 1.04s
